In [71]:
import pandas as pd
import numpy as np
import plotly.express as px

In [72]:
df_ia = pd.read_csv("../data/raw/occ_level.csv")
df_cw_soc = pd.read_csv("../data/raw/crosswalks/soc_data.csv", dtype=str)
df_cw_cps = pd.read_csv("../data/raw/crosswalks/cps_2020_data.csv", dtype=str)
df_bls = pd.read_csv("../data/raw/bls_oews_national.csv", dtype=str)
### df_cw = pd.read_csv("../data/raw/census_crosswalk.csv", dtype=str)
df_cps = pd.read_csv("../data/raw/cps_00001.csv")


In [73]:
if "O_GROUP" in df_bls.columns:
    df_bls = df_bls[df_bls["O_GROUP"] == "detailed"].copy()

In [74]:
df_bls["TOT_EMP"] = pd.to_numeric(df_bls["TOT_EMP"].str.replace(",", ""), errors="coerce")
df_bls["A_MEDIAN"] = pd.to_numeric(df_bls["A_MEDIAN"].str.replace(",", ""), errors="coerce")
df_bls["SOC_6dig"] = df_bls["OCC_CODE"].str.strip()

In [75]:
df_bls_limpio = df_bls[["SOC_6dig", "OCC_TITLE", "TOT_EMP", "A_MEDIAN"]].dropna(subset=["SOC_6dig"])

# Problemas

In [76]:
df = pd.read_csv(
    '../data/raw/census_crosswalk.csv',
    usecols=[0, 1, 2, 3],
    skiprows=8,
    header=None,
    dtype=str,
)

df.columns = ['Major_Group', 'Census_Title', 'Census_Code', 'SOC_Code']

In [77]:
df = df.dropna(how='all').copy()

In [78]:
df['Major_Group'] = df['Major_Group'].str.rstrip(':').ffill()
df_clean = df[df['Census_Code'].str.match(r'^\d{4}$', na=False)].copy()
for col in df_clean.columns:
  df_clean[col] = df_clean[col].str.strip()

df_cw = df_clean.reset_index(drop=True)

df_cw

,Major_Group,Census_Title,Census_Code,SOC_Code
0,"Management, Business, Science, and Arts Occup...",Chief executives,0010,11-1011
1,"Management, Business, Science, and Arts Occup...",General and operations managers,0020,11-1021
2,"Management, Business, Science, and Arts Occup...",Legislators,0030,11-1031
3,"Management, Business, Science, and Arts Occup...",Advertising and promotions managers,0040,11-2011
4,"Management, Business, Science, and Arts Occup...",Marketing managers,0051,11-2021
...,...,...,...,...
565,Military Specific Occupations,Military officer special and tactical operatio...,9800,55-1010
566,Military Specific Occupations,First-line enlisted military supervisors,9810,55-2010
567,Military Specific Occupations,Military enlisted tactical operations and air/...,9825,55-3010
568,Military Specific Occupations,"Military, rank not Specified",9830,none


In [82]:
df_cps['OCC_clean'] = (
    df_cps['OCC'].fillna(0).astype(int).astype(str).str.zfill(4)
)
df_cw['Census_Code'] = df_cw['Census_Code'].astype(str).str.zfill(4)

In [84]:
df_merged = pd.merge(
    df_cps,
    df_cw,
    left_on='OCC_clean',
    right_on='Census_Code',
    how='left',
)

df_merged

,YEAR,SERIAL,MONTH,HWTFINL,CPSID,ASECFLAG,ASECWTH,PERNUM,WTFINL,CPSIDP,...,EMPSTAT,OCC,OCC2010,DURUNEMP,EDUC,OCC_clean,Major_Group,Census_Title,Census_Code,SOC_Code
0,2021,1,1,1805.3669,20191100000100,NaN,NaN,1,1805.3669,20191100000101,...,32,0,9999,999,81,0000,NaN,NaN,NaN,NaN
1,2021,1,1,1805.3669,20191100000100,NaN,NaN,2,2336.9526,20191100000102,...,10,4150,4150,999,73,4150,Service Occupations,"Hosts and hostesses, restaurant, lounge, and c...",4150,35-9031
2,2021,1,1,1805.3669,20191100000100,NaN,NaN,3,2129.3099,20191100000103,...,21,4110,4110,39,111,4110,Service Occupations,Waiters and waitresses,4110,35-3031
3,2021,2,1,1903.8219,20201100000100,NaN,NaN,1,1903.8219,20201100000101,...,36,0,9999,999,123,0000,NaN,NaN,NaN,NaN
4,2021,3,1,2738.9084,20191200000300,NaN,NaN,1,1570.4995,20191200000301,...,10,3602,4610,999,73,3602,Service Occupations,Personal care aides,3602,31-1122
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7724466,2026,69172,10,5158.6941,20250706859600,NaN,NaN,2,5501.4433,20250706859602,...,10,850,850,999,111,0850,"Management, Business, Science, and Arts Occup...",Personal financial advisors,0850,13-2052
7724467,2026,69173,10,4778.5221,20250806883900,NaN,NaN,1,4778.5221,20250806883901,...,10,3930,3930,999,123,3930,Service Occupations,Security guards and gambling surveillance offi...,3930,33-9030
7724468,2026,69173,10,4778.5221,20250806883900,NaN,NaN,2,5557.7384,20250806883902,...,10,3603,3600,999,92,3603,Service Occupations,Nursing assistants,3603,31-1131
7724469,2026,69174,10,4764.5418,20251107001400,NaN,NaN,1,4764.5418,20251107001401,...,10,2014,2010,999,123,2014,"Management, Business, Science, and Arts Occup...","Social workers, all other",2014,21-1029


In [85]:
df_merged = df_merged.drop(columns=['OCC_clean', 'Census_Code'])

In [86]:
print(f'Filas totales: {len(df_merged)}')
print('Ocupaciones sin cruce (incluye código 0000 / NIU):')
print(df_merged[df_merged['Census_Title'].isna()]['OCC'].value_counts())

Filas totales: 7724471
Ocupaciones sin cruce (incluye código 0000 / NIU):
OCC
0       3893793
9840        465
Name: count, dtype: int64
